In [1]:
!pip install -q pymupdf sentence-transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 50.6 MB/s eta 0:00:00


In [2]:
import fitz
import re
import torch
import numpy as np
import pandas as pd

from google.colab import files

from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering
)

from sentence_transformers import SentenceTransformer, util

In [3]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cpu


In [4]:
def extract_text_from_pdf(pdf_path):

    document = fitz.open(pdf_path)

    extracted_pages = []

    for page_number in range(len(document)):

        page = document[page_number]

        text = page.get_text("text")

        extracted_pages.append(text)

    document.close()

    full_text = "\n".join(extracted_pages)

    return full_text

In [5]:
def clean_text(text):

    # Replace line-breaking hyphenation if present
    text = re.sub(r'-\n', '', text)

    # Replace remaining newlines with spaces
    text = text.replace('\n', ' ')

    # Remove long groups of symbols that appear in the Hard PDF
    text = re.sub(r'[_$%&()]{2,}', ' ', text)

    # Reduce the same punctuation repeated 3 or more times
    text = re.sub(r'([^\w\s])\1{2,}', r'\1', text)

    # Remove groups consisting mainly of unusual special characters
    text = re.sub(r'[^\w\s.,?!\'":;-]{3,}', ' ', text)

    # Remove unnecessary spaces before punctuation
    text = re.sub(r'\s+([.,?!;:])', r'\1', text)

    # Convert multiple spaces into one space
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [6]:
def split_into_chunks(text, chunk_size=180, overlap=40):

    words = text.split()

    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(words[start:end])

        if len(chunk.strip()) > 0:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [7]:
print("Loading Sentence Transformer model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Sentence Transformer loaded successfully.")

Loading Sentence Transformer model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence Transformer loaded successfully.


In [8]:
print("Loading DistilBERT Question Answering model...")

qa_model_name = "distilbert-base-cased-distilled-squad"

qa_tokenizer = AutoTokenizer.from_pretrained(
    qa_model_name
)

qa_model = AutoModelForQuestionAnswering.from_pretrained(
    qa_model_name
)

qa_model.to(device)

qa_model.eval()

print("Question Answering model loaded successfully.")

Loading DistilBERT Question Answering model...


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  261MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Question Answering model loaded successfully.


In [9]:
def run_qa_model(question, context):

    inputs = qa_tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = qa_model(**inputs)

    start_logits = outputs.start_logits[0]
    end_logits = outputs.end_logits[0]

    start_probs = torch.softmax(
        start_logits,
        dim=0
    )

    end_probs = torch.softmax(
        end_logits,
        dim=0
    )

    best_score = -1
    best_start = 0
    best_end = 0

    max_answer_length = 30

    for start_index in range(len(start_probs)):

        max_end = min(
            start_index + max_answer_length,
            len(end_probs)
        )

        for end_index in range(
            start_index,
            max_end
        ):

            score = (
                start_probs[start_index]
                * end_probs[end_index]
            ).item()

            if score > best_score:

                best_score = score
                best_start = start_index
                best_end = end_index

    answer_tokens = inputs["input_ids"][0][
        best_start:best_end + 1
    ]

    answer = qa_tokenizer.decode(
        answer_tokens,
        skip_special_tokens=True
    )

    return {
        "answer": answer.strip(),
        "score": best_score
    }

In [10]:
test_question = "What is the capital city of Bangladesh?"

test_context = """
Bangladesh is a country in South Asia.
The capital city of Bangladesh is Dhaka.
Bangladesh gained independence in 1971.
"""

test_result = run_qa_model(
    test_question,
    test_context
)

print("Question:", test_question)
print("Answer:", test_result["answer"])
print("Confidence:", test_result["score"])

Question: What is the capital city of Bangladesh?
Answer: Dhaka
Confidence: 0.9811953902244568


In [11]:
def retrieve_relevant_chunks(
    question,
    chunks,
    chunk_embeddings,
    top_k=3
):

    question_embedding = embedding_model.encode(
        question,
        convert_to_tensor=True
    )

    similarities = util.cos_sim(
        question_embedding,
        chunk_embeddings
    )[0]

    top_results = torch.topk(
        similarities,
        k=min(top_k, len(chunks))
    )

    retrieved = []

    for score, index in zip(
        top_results.values,
        top_results.indices
    ):

        retrieved.append({
            "chunk": chunks[index.item()],
            "similarity": score.item()
        })

    return retrieved

In [12]:
def answer_question(question, top_k=3):

    retrieved_chunks = retrieve_relevant_chunks(
        question,
        chunks,
        chunk_embeddings,
        top_k=top_k
    )

    candidates = []

    for item in retrieved_chunks:

        context = item["chunk"]

        result = run_qa_model(
            question,
            context
        )

        qa_score = float(result["score"])
        similarity = float(item["similarity"])

        final_score = (
            0.8 * qa_score
            + 0.2 * similarity
        )

        candidates.append({
            "answer": result["answer"],
            "qa_score": qa_score,
            "similarity": similarity,
            "final_score": final_score,
            "context": context
        })

    best_answer = max(
        candidates,
        key=lambda x: x["final_score"]
    )

    return best_answer

In [40]:
from google.colab import files

print("Please upload the Easy PDF")

uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

print("Uploaded PDF:", pdf_path)

Please upload the Easy PDF


Saving Test PDF (Easy).pdf to Test PDF (Easy).pdf
Uploaded PDF: Test PDF (Easy).pdf


In [41]:
process_pdf(pdf_path)

STEP 1: Extracting text from PDF
Text extraction completed.
Raw text length: 2481

STEP 2: Preprocessing text
Text preprocessing completed.
Cleaned text length: 2438

STEP 3: Creating text chunks
Number of chunks: 3

STEP 4: Creating semantic embeddings


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


PDF processing completed successfully!


In [17]:
def process_pdf(pdf_path):

    global raw_text
    global cleaned_text
    global chunks
    global chunk_embeddings

    print("=" * 70)
    print("STEP 1: Extracting text from PDF")
    print("=" * 70)

    raw_text = extract_text_from_pdf(pdf_path)

    print("Text extraction completed.")
    print("Raw text length:", len(raw_text))

    print("\n" + "=" * 70)
    print("STEP 2: Preprocessing text")
    print("=" * 70)

    cleaned_text = clean_text(raw_text)

    print("Text preprocessing completed.")
    print("Cleaned text length:", len(cleaned_text))

    print("\n" + "=" * 70)
    print("STEP 3: Creating text chunks")
    print("=" * 70)

    chunks = split_into_chunks(cleaned_text)

    print("Number of chunks:", len(chunks))

    print("\n" + "=" * 70)
    print("STEP 4: Creating semantic embeddings")
    print("=" * 70)

    chunk_embeddings = embedding_model.encode(
        chunks,
        convert_to_tensor=True,
        show_progress_bar=True
    )

    print("\nPDF processing completed successfully!")

In [18]:
process_pdf(pdf_path)

STEP 1: Extracting text from PDF
Text extraction completed.
Raw text length: 2481

STEP 2: Preprocessing text
Text preprocessing completed.
Cleaned text length: 2438

STEP 3: Creating text chunks
Number of chunks: 3

STEP 4: Creating semantic embeddings


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


PDF processing completed successfully!


In [19]:
print("RAW EXTRACTED TEXT")
print("=" * 80)

print(raw_text)

RAW EXTRACTED TEXT
Bangladesh is a beautiful country in South Asia, known for its rich cultural heritage, fertile land, 
and vast river systems. The capital city of Bangladesh is Dhaka. Dhaka is not only the political 
center but also the economic and cultural heart of the nation. It is one of the most densely 
populated cities in the world and plays a major role in the country’s development. 
 
One of the most important natural features of Bangladesh is the Padma River, which is often 
referred to as the lifeline of the country. This river is crucial for transportation, agriculture, 
fishing, and daily life. Many people depend on it for their livelihoods, and it supports the fertile 
land that makes Bangladesh suitable for farming. 
 
Bangladesh gained its independence in 1971. This historic event is remembered every year as 
Independence Day and is a source of great pride for the people of Bangladesh. The struggle for 
independence played a key role in shaping the nation’s identity a

In [20]:
print("CLEANED TEXT")
print("=" * 80)

print(cleaned_text)

CLEANED TEXT
Bangladesh is a beautiful country in South Asia, known for its rich cultural heritage, fertile land, and vast river systems. The capital city of Bangladesh is Dhaka. Dhaka is not only the political center but also the economic and cultural heart of the nation. It is one of the most densely populated cities in the world and plays a major role in the country’s development. One of the most important natural features of Bangladesh is the Padma River, which is often referred to as the lifeline of the country. This river is crucial for transportation, agriculture, fishing, and daily life. Many people depend on it for their livelihoods, and it supports the fertile land that makes Bangladesh suitable for farming. Bangladesh gained its independence in 1971. This historic event is remembered every year as Independence Day and is a source of great pride for the people of Bangladesh. The struggle for independence played a key role in shaping the nation’s identity and unity. The offici

In [21]:
print("PREPROCESSING STATISTICS")
print("=" * 50)

print("Characters before cleaning:", len(raw_text))
print("Characters after cleaning :", len(cleaned_text))

removed = len(raw_text) - len(cleaned_text)

print("Characters removed        :", removed)
print("Number of chunks          :", len(chunks))

PREPROCESSING STATISTICS
Characters before cleaning: 2481
Characters after cleaning : 2438
Characters removed        : 43
Number of chunks          : 3


In [22]:
print("NUMBER OF CHUNKS:", len(chunks))

for i, chunk in enumerate(chunks):

    print("\n" + "=" * 70)
    print("CHUNK", i + 1)
    print("=" * 70)

    print(chunk)

NUMBER OF CHUNKS: 3

CHUNK 1
Bangladesh is a beautiful country in South Asia, known for its rich cultural heritage, fertile land, and vast river systems. The capital city of Bangladesh is Dhaka. Dhaka is not only the political center but also the economic and cultural heart of the nation. It is one of the most densely populated cities in the world and plays a major role in the country’s development. One of the most important natural features of Bangladesh is the Padma River, which is often referred to as the lifeline of the country. This river is crucial for transportation, agriculture, fishing, and daily life. Many people depend on it for their livelihoods, and it supports the fertile land that makes Bangladesh suitable for farming. Bangladesh gained its independence in 1971. This historic event is remembered every year as Independence Day and is a source of great pride for the people of Bangladesh. The struggle for independence played a key role in shaping the nation’s identity and u

In [23]:
question = "What is the capital city of Bangladesh?"

retrieved = retrieve_relevant_chunks(
    question,
    chunks,
    chunk_embeddings,
    top_k=3
)

for i, item in enumerate(retrieved, start=1):

    print("\n" + "=" * 70)
    print("Retrieved Chunk:", i)
    print("Similarity:", round(item["similarity"], 4))
    print("=" * 70)

    print(item["chunk"])


Retrieved Chunk: 1
Similarity: 0.6152
Bangladesh is a beautiful country in South Asia, known for its rich cultural heritage, fertile land, and vast river systems. The capital city of Bangladesh is Dhaka. Dhaka is not only the political center but also the economic and cultural heart of the nation. It is one of the most densely populated cities in the world and plays a major role in the country’s development. One of the most important natural features of Bangladesh is the Padma River, which is often referred to as the lifeline of the country. This river is crucial for transportation, agriculture, fishing, and daily life. Many people depend on it for their livelihoods, and it supports the fertile land that makes Bangladesh suitable for farming. Bangladesh gained its independence in 1971. This historic event is remembered every year as Independence Day and is a source of great pride for the people of Bangladesh. The struggle for independence played a key role in shaping the nation’s iden

In [24]:
question = "What is the capital city of Bangladesh?"

result = answer_question(question)

print("=" * 70)
print("QUESTION:")
print(question)

print("\nANSWER:")
print(result["answer"])

print("\nQA CONFIDENCE:")
print(round(result["qa_score"], 4))

print("\nRETRIEVAL SIMILARITY:")
print(round(result["similarity"], 4))

print("=" * 70)

QUESTION:
What is the capital city of Bangladesh?

ANSWER:
Dhaka

QA CONFIDENCE:
0.9723

RETRIEVAL SIMILARITY:
0.6152


In [25]:
question = "In which year did Bangladesh gain independence?"

result = answer_question(question)

print("Question:", question)
print("Answer:", result["answer"])
print("Confidence:", round(result["qa_score"], 4))

Question: In which year did Bangladesh gain independence?
Answer: 1971
Confidence: 0.9957


In [26]:
question = "Which river is considered the lifeline of Bangladesh?"

result = answer_question(question)

print("Question:", question)
print("Answer:", result["answer"])
print("Confidence:", round(result["qa_score"], 4))

Question: Which river is considered the lifeline of Bangladesh?
Answer: Padma River
Confidence: 0.7415


In [27]:
questions = [
    "What is the capital city of Bangladesh?",
    "Which river is considered the lifeline of Bangladesh?",
    "In which year did Bangladesh gain independence?",
    "What is the official language of Bangladesh?",
    "What is the name of the world's largest mangrove forest located in Bangladesh?",
    "Which currency is used in Bangladesh?",
    "What is the national animal of Bangladesh?"
]

print("Total questions:", len(questions))

Total questions: 7


In [28]:
for number, question in enumerate(questions, start=1):

    result = answer_question(question)

    print("=" * 80)

    print(f"Question {number}: {question}")

    print(f"Answer: {result['answer']}")

    print(
        f"Confidence: {result['qa_score']:.4f}"
    )

print("=" * 80)

Question 1: What is the capital city of Bangladesh?
Answer: Dhaka
Confidence: 0.9723
Question 2: Which river is considered the lifeline of Bangladesh?
Answer: Padma River
Confidence: 0.7415
Question 3: In which year did Bangladesh gain independence?
Answer: 1971
Confidence: 0.9957
Question 4: What is the official language of Bangladesh?
Answer: Bengali
Confidence: 0.9679
Question 5: What is the name of the world's largest mangrove forest located in Bangladesh?
Answer: Sundarbans
Confidence: 0.8658
Question 6: Which currency is used in Bangladesh?
Answer: Bangladeshi Taka
Confidence: 0.9056
Question 7: What is the national animal of Bangladesh?
Answer: Royal Bengal Tiger
Confidence: 0.8351


In [45]:
results_data = []

for number, question in enumerate(
    questions,
    start=1
):

    result = answer_question(question)

    results_data.append({
        "No.": number,
        "Question": question,
        "Answer": result["answer"],
        "Confidence": round(
            result["qa_score"],
            4
        )
    })


results_df = pd.DataFrame(results_data)

pd.set_option("display.max_colwidth", None)

print("Easy PDF Output")
display(results_df)

Easy PDF Output


,No.,Question,Answer,Confidence
0,1,What is the capital city of Bangladesh?,Dhaka,0.9739
1,2,Which river is considered the lifeline of Bangladesh?,Padma River,0.7876
2,3,In which year did Bangladesh gain independence?,1971,0.9952
3,4,What is the official language of Bangladesh?,Bengali,0.9647
4,5,What is the name of the world's largest mangrove forest located in Bangladesh?,Sundarbans,0.8910
5,6,Which currency is used in Bangladesh?,Bangladeshi Taka,0.9118
6,7,What is the national animal of Bangladesh?,Royal Bengal Tiger,0.8507


In [32]:
print("=" * 70)
print("PDF QUESTION ANSWERING SYSTEM")
print("=" * 70)
print("Ask a question about the uploaded PDF.")
print("Type 'exit' to stop.")

while True:

    question = input("\nEnter your question: ")

    if question.lower().strip() == "exit":
        print("Question Answering System stopped.")
        break

    if question.strip() == "":
        print("Please enter a valid question.")
        continue

    result = answer_question(question)

    print("\nAnswer:", result["answer"])
    print(
        "Confidence:",
        round(result["qa_score"], 4)
    )

PDF QUESTION ANSWERING SYSTEM
Ask a question about the uploaded PDF.
Type 'exit' to stop.

Enter your question: What is the official language of Bangladesh?

Answer: Bengali
Confidence: 0.9679

Enter your question: exit
Question Answering System stopped.


In [42]:
print("Please upload the Hard PDF")

uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

print("Uploaded PDF:", pdf_path)

Please upload the Hard PDF


Saving Test PDF (Hard).pdf to Test PDF (Hard).pdf
Uploaded PDF: Test PDF (Hard).pdf


In [43]:
process_pdf(pdf_path)

STEP 1: Extracting text from PDF
Text extraction completed.
Raw text length: 4819

STEP 2: Preprocessing text
Text preprocessing completed.
Cleaned text length: 2307

STEP 3: Creating text chunks
Number of chunks: 3

STEP 4: Creating semantic embeddings


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


PDF processing completed successfully!


In [36]:
print("HARD PDF - BEFORE PREPROCESSING")
print("=" * 80)

print(raw_text[:4000])

HARD PDF - BEFORE PREPROCESSING
Bangladesh(_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&
&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&
&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%
%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%
%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$
$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((
((((((_. is a beautiful country in South Asia, known for its rich cultural heritage, fertile land, and 
vast river systems. The capital city of Bangladesh is Dhaka. Dhaka is not only the political 
center but also the economic and cultural heart of the nation. It is one of the most densely 
populated cities in the world and plays a major role in the country’s development. 
 
One of the most important natural features of Bangladesh is the 
(_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&()
))))))))

In [37]:
print("HARD PDF - AFTER PREPROCESSING")
print("=" * 80)

print(cleaned_text[:4000])

HARD PDF - AFTER PREPROCESSING
Bangladesh. is a beautiful country in South Asia, known for its rich cultural heritage, fertile land, and vast river systems. The capital city of Bangladesh is Dhaka. Dhaka is not only the political center but also the economic and cultural heart of the nation. It is one of the most densely populated cities in the world and plays a major role in the country’s development. One of the most important natural features of Bangladesh is the. Padma River, which is often referred to as the lifeline of the country. This river is crucial for transportation, agriculture, fishing, and daily life. Many people depend on it for their livelihoods, and it supports the fertile land that makes Bangladesh suitable for farming. Bangladesh gained its independence in 1971. This historic event is remembered every year as Independence Day and is a source of great pride for the people of Bangladesh. The struggle for independence played a key role in shaping the nation’s identity a

In [38]:
print("HARD PDF PREPROCESSING STATISTICS")
print("=" * 60)

print(
    "Characters before preprocessing:",
    len(raw_text)
)

print(
    "Characters after preprocessing :",
    len(cleaned_text)
)

print(
    "Characters removed             :",
    len(raw_text) - len(cleaned_text)
)

print(
    "Number of generated chunks     :",
    len(chunks)
)

HARD PDF PREPROCESSING STATISTICS
Characters before preprocessing: 4819
Characters after preprocessing : 2307
Characters removed             : 2512
Number of generated chunks     : 3


In [46]:
hard_results = []

for number, question in enumerate(
    questions,
    start=1
):

    result = answer_question(question)

    hard_results.append({
        "No.": number,
        "Question": question,
        "Answer": result["answer"],
        "Confidence": round(
            result["qa_score"],
            4
        )
    })


hard_results_df = pd.DataFrame(hard_results)

pd.set_option("display.max_colwidth", None)

print("Hard PDF Output")
display(hard_results_df)

Hard PDF Output


,No.,Question,Answer,Confidence
0,1,What is the capital city of Bangladesh?,Dhaka,0.9739
1,2,Which river is considered the lifeline of Bangladesh?,Padma River,0.7876
2,3,In which year did Bangladesh gain independence?,1971,0.9952
3,4,What is the official language of Bangladesh?,Bengali,0.9647
4,5,What is the name of the world's largest mangrove forest located in Bangladesh?,Sundarbans,0.8910
5,6,Which currency is used in Bangladesh?,Bangladeshi Taka,0.9118
6,7,What is the national animal of Bangladesh?,Royal Bengal Tiger,0.8507
